# Yelp Dataset — Data Preparation

**Goal**: Download the Yelp dataset, join reviews with business metadata,
sample 1 million rows, apply a temporal split, and save train/val/test CSVs.

**Files used from Kaggle:**
- `yelp_academic_dataset_review.json` — ratings (user_id, business_id, stars, date)
- `yelp_academic_dataset_business.json` — business metadata (city, state, categories, stars, review_count)

**Output files**
```
yelp/data/
  train.csv    (~70% of 1M)
  val.csv      (~15% of 1M)
  test.csv     (~15% of 1M)
```

**What makes Yelp unique vs MovieLens and Amazon:**
- Business metadata available (city, state, categories) — new feature types
- Categories is multi-label like MovieLens genres (e.g. "Restaurants, Italian, Pizza")
- City and state are low-cardinality categoricals — clean OHE baseline
- Ratings are 1–5 integer stars (no half-stars like MovieLens)


## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Download dataset via kagglehub
STEP 3 — Load review data
STEP 4 — Load business metadata
STEP 5 — Join reviews with business metadata
STEP 6 — Clean & filter
STEP 7 — Sample 1 million rows
STEP 8 — Temporal train / val / test split (70/15/15)
STEP 9 — Save to disk
STEP 10 — Sanity checks
```


---
## Step 1 — Imports & Paths

In [ ]:
import os, json, warnings
import numpy  as np
import pandas as pd
warnings.filterwarnings('ignore')


In [ ]:
BASE_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp"
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\yelp\\data"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Base dir : {BASE_DIR}")
print(f"Data dir : {DATA_DIR}")


---
## Step 2 — Download Dataset

Dataset: yelp-dataset/yelp-dataset
Run once — kagglehub caches locally after the first download.


In [ ]:
import kagglehub
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")
print(f"Downloaded to: {path}")
print("\nFiles available:")
for f in sorted(os.listdir(path)):
    size_mb = os.path.getsize(os.path.join(path, f)) / 1e6
    print(f"  {f:<55}  {size_mb:.1f} MB")


---
## Step 3 — Load Review Data

The review file is a JSON-lines format — one JSON object per line.
We read only the columns we need to keep memory usage low.

Columns kept: user_id, business_id, stars, date


In [ ]:
REVIEW_FILE = os.path.join(path, 'yelp_academic_dataset_review.json')

# Read JSON lines — each line is a complete JSON object
# We only keep the 4 columns we need — the review text etc are not used
review_rows = []
KEEP_REVIEW = {'user_id', 'business_id', 'stars', 'date'}

print("Reading review file...")
with open(REVIEW_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        review_rows.append({k: obj[k] for k in KEEP_REVIEW if k in obj})
        if (i + 1) % 1_000_000 == 0:
            print(f"  Read {i+1:,} lines...")

df_reviews = pd.DataFrame(review_rows)
print(f"\nReview data shape : {df_reviews.shape}")
print(f"Columns           : {list(df_reviews.columns)}")
df_reviews.head(3)


In [ ]:
# Check rating distribution
print("Star rating distribution:")
print(df_reviews['stars'].value_counts().sort_index())
print(f"\nDate range: {df_reviews['date'].min()} -> {df_reviews['date'].max()}")
print(f"Unique users     : {df_reviews['user_id'].nunique():,}")
print(f"Unique businesses: {df_reviews['business_id'].nunique():,}")


---
## Step 4 — Load Business Metadata

Business data provides city, state, and categories — the new categorical
features that distinguish Yelp from Amazon.

**categories** is a comma-separated string like "Restaurants, Italian, Pizza"
— same structure as MovieLens genres, will be mean-pooled in embeddings.


In [ ]:
BUSINESS_FILE = os.path.join(path, 'yelp_academic_dataset_business.json')

# Columns we need from business data
KEEP_BUSINESS = {'business_id', 'city', 'state', 'categories',
                 'stars', 'review_count'}

business_rows = []
with open(BUSINESS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        business_rows.append({k: obj.get(k) for k in KEEP_BUSINESS})

df_business = pd.DataFrame(business_rows)
# Rename to avoid collision with review stars
df_business = df_business.rename(columns={
    'stars'        : 'business_avg_stars',
    'review_count' : 'business_review_count'
})

print(f"Business data shape : {df_business.shape}")
print(f"Unique businesses   : {df_business['business_id'].nunique():,}")
print(f"Unique cities       : {df_business['city'].nunique():,}")
print(f"Unique states       : {df_business['state'].nunique():,}")
df_business.head(3)


In [ ]:
# Sample of categories to understand the format
print("Sample categories:")
sample = df_business['categories'].dropna().head(10)
for s in sample:
    print(f"  {s}")


---
## Step 5 — Join Reviews with Business Metadata

Left join reviews onto business data using business_id.
This adds city, state, categories, business_avg_stars, and business_review_count
to every review row.

We use a left join so no review rows are dropped — businesses not found
in the business file will get NaN values (handled in cleaning).


In [ ]:
df = df_reviews.merge(df_business, on='business_id', how='left')

print(f"Shape after join : {df.shape}")
print(f"Missing values:")
print(df.isna().sum())


---
## Step 6 — Clean & Filter

**Cleaning steps:**
1. Drop rows with missing stars (target) or missing user/business IDs
2. Ensure stars is in valid range (1–5)
3. Parse date to datetime
4. Fill missing categorical metadata with 'Unknown'
5. Deduplicate (user, business) pairs — keep most recent review
6. Normalise categories separator — Yelp uses ", " (comma-space) not "|"
   We convert to "|" for consistency with MovieLens pipeline


In [ ]:
# Step 6a: Drop missing targets and keys
before = len(df)
df = df.dropna(subset=['stars', 'user_id', 'business_id'])
print(f"Dropped {before - len(df):,} rows with missing stars/user/business")


In [ ]:
# Step 6b: Valid rating range (1-5 integer stars)
before = len(df)
df = df[df['stars'].between(1, 5)]
print(f"Dropped {before - len(df):,} rows with out-of-range stars")
print(f"Star distribution: {df['stars'].value_counts().sort_index().to_dict()}")


In [ ]:
# Step 6c: Parse date
df['datetime'] = pd.to_datetime(df['date'], errors='coerce')
before = len(df)
df = df.dropna(subset=['datetime'])
print(f"Dropped {before - len(df):,} rows with unparseable dates")
print(f"Date range: {df['datetime'].min()} -> {df['datetime'].max()}")


In [ ]:
# Step 6d: Fill missing categorical metadata
df['city']       = df['city'].fillna('Unknown')
df['state']      = df['state'].fillna('Unknown')
df['categories'] = df['categories'].fillna('Unknown')

# Normalise categories separator: ", " -> "|" for consistency with MovieLens
df['categories'] = df['categories'].str.replace(', ', '|', regex=False)

print("Sample categories after normalisation:")
print(df['categories'].head(5).tolist())


In [ ]:
# Step 6e: Deduplicate (user, business) pairs — keep most recent
before = len(df)
df = df.sort_values('datetime').drop_duplicates(
    subset=['user_id', 'business_id'], keep='last'
)
print(f"Dropped {before - len(df):,} duplicate (user, business) pairs")
print(f"Clean dataset size: {len(df):,} rows")


In [ ]:
# Step 6f: Drop columns we no longer need
df = df.drop(columns=['date'], errors='ignore')

print(f"Final columns: {list(df.columns)}")
print(f"Final shape  : {df.shape}")


---
## Step 7 — Sample 1 Million Rows

Sample before splitting to keep the time distribution representative.
Same seed as MovieLens and Amazon for reproducibility.


In [ ]:
SAMPLE_SIZE = 1_000_000
SEED        = 42

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_SIZE:,} rows from {len(df):,} total")
else:
    print(f"Dataset has {len(df):,} rows — using all of them")

print(f"Working dataset : {len(df):,} rows")
print(f"Date range      : {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Unique users    : {df['user_id'].nunique():,}")
print(f"Unique businesses: {df['business_id'].nunique():,}")
print(f"Unique cities   : {df['city'].nunique():,}")
print(f"Unique states   : {df['state'].nunique():,}")


---
## Step 8 — Temporal Train / Val / Test Split (70 / 15 / 15)

Percentile-based temporal split — same approach as Amazon.
Guarantees consistent proportions regardless of date distribution.

All rows in train are chronologically earlier than all rows in val,
and all val rows are earlier than all test rows.


In [ ]:
df_sorted = df.sort_values('datetime')
n         = len(df_sorted)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_cutoff = df_sorted.iloc[train_end]['datetime']
val_cutoff   = df_sorted.iloc[val_end]['datetime']

print(f"Train cutoff : {train_cutoff}")
print(f"Val cutoff   : {val_cutoff}")


In [ ]:
df_train = df[df['datetime'] <  train_cutoff].copy()
df_val   = df[(df['datetime'] >= train_cutoff) &
              (df['datetime'] <  val_cutoff)].copy()
df_test  = df[df['datetime'] >= val_cutoff].copy()

print(f"Train : {len(df_train):,} rows  ({len(df_train)/len(df)*100:.1f}%)")
print(f"Val   : {len(df_val):,} rows   ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test  : {len(df_test):,} rows  ({len(df_test)/len(df)*100:.1f}%)")


In [ ]:
# Verify no overlap
assert len(set(df_train.index) & set(df_val.index))  == 0
assert len(set(df_train.index) & set(df_test.index)) == 0
assert len(set(df_val.index)   & set(df_test.index)) == 0
assert len(df_train) + len(df_val) + len(df_test)    == len(df)
print("No overlaps — split is clean ✓")


In [ ]:
# Rating distribution across splits
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    dist = split['stars'].value_counts().sort_index()
    pct  = (dist / len(split) * 100).round(1)
    print(f"\n{name} ({len(split):,} rows):")
    for star, count in dist.items():
        print(f"  {star:.0f} star : {count:,}  ({pct[star]:.1f}%)")


---
## Step 9 — Save to Disk

In [ ]:
train_path = os.path.join(DATA_DIR, 'train.csv')
val_path   = os.path.join(DATA_DIR, 'val.csv')
test_path  = os.path.join(DATA_DIR, 'test.csv')

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path,     index=False)
df_test.to_csv(test_path,   index=False)

print("Files saved:")
for fpath in [train_path, val_path, test_path]:
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {os.path.basename(fpath):<15}  {size_mb:.1f} MB")


---
## Step 10 — Sanity Checks

In [ ]:
train_check = pd.read_csv(train_path)
val_check   = pd.read_csv(val_path)
test_check  = pd.read_csv(test_path)

print("Reloaded shapes:")
print(f"  train : {train_check.shape}")
print(f"  val   : {val_check.shape}")
print(f"  test  : {test_check.shape}")
print(f"  Columns: {list(train_check.columns)}")


In [ ]:
print("=" * 60)
print("YELP — DATA PREPARATION COMPLETE")
print("=" * 60)
print(f"  Total rows       : {len(df_train)+len(df_val)+len(df_test):,}")
print(f"  Train            : {len(df_train):,}  (70%)")
print(f"  Val              : {len(df_val):,}   (15%)")
print(f"  Test             : {len(df_test):,}   (15%)")
print(f"  Target           : stars (1-5 integer)")
print(f"  Categoricals     : user_id, business_id, city, state, categories")
print(f"  Split type       : Temporal (70/15/15 percentile)")
print(f"  Unique cities    : {df['city'].nunique():,}")
print(f"  Unique states    : {df['state'].nunique():,}")
print("=" * 60)
print(f"\nNext step: run Yelp_01_FeatureEngineering.ipynb")
